In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from pymargins import GComputation

rng = np.random.default_rng(42)
n = 1000
df = pd.DataFrame({
    "age": rng.integers(18, 80, size=n),
    "treat": rng.binomial(1, 0.4, size=n),
    "x1": rng.normal(size=n),
})
logit_p = 1 / (1 + np.exp(-(-1.0 + 0.04 * df["age"] + 0.5 * df["treat"] + 0.3 * df["x1"])))
df["y"] = rng.binomial(1, logit_p)

model = smf.logit("y ~ treat*x1 + age", data=df).fit(disp=0)
est = GComputation(model, at="overall", scale="response", method="delta")

est.predict()                                    # average adjusted prediction
est.predict(atexog={"treat": [0, 1]})            # standardized rates
est.contrasts(
    scenarios=[{"atexog": {"treat": 0}}, {"atexog": {"treat": 1}}],
    contrasts=[-1, 1],
)                                                  # marginal risk difference
est.dydx("age")                                  # AME

GraphResult(estimate=array(0.00613843, dtype=float32), std_error=array(0.00062575, dtype=float32), conf_int_lower=array(0.00491198, dtype=float32), conf_int_upper=array(0.00736489, dtype=float32), labels=['age'], method='delta', level=0.95, ci='wald', scale='response', at='overall', plan=Plan(package_version='0.4.0', plan_hash='cfa144cb78ee2032372feed1f79534c5d912e0fff4da96459a05356adbe7458e', node_kinds=('input',), node_hashes=('4966aed573a2b67cbc797167dbc93d2d563ef91d77888f97dc96041bc096e763',), at='overall', scale='response', method_declared='delta', method_resolved='delta', method_resolution_reason='user-specified', vcov=None, ci='wald', level=0.95, B=1000, n_sim=4000, seed=None, gradient_backend='autodiff', fd_step=1e-06, data_fingerprint='c13175e403ec591f64c4df2d00aa4bbfb655cab6a9f161bf8ca9a52424d612aa', weights_fingerprint=None, unhashable_callable=False, population_note=None, constants_overrides=()), population_note=None, n_obs=1000, estimand_metadata={'kind': 'slope', 'variabl

In [2]:
# Requires: pip install lifelines
from lifelines import CoxPHFitter

rng = np.random.default_rng(7)
n = 800
df_surv = pd.DataFrame({
    "time": rng.exponential(50, size=n),
    "event": rng.binomial(1, 0.8, size=n),
    "treat": rng.binomial(1, 0.4, size=n),
    "age": rng.normal(50, 10, size=n),
})

cph = CoxPHFitter().fit(df_surv, "time", "event")

from pymargins.adapters import LifelinesCoxPHAdapter

est = GComputation(
    cph,
    adapter=LifelinesCoxPHAdapter(cph, training_data=df_surv),
    at="overall",
    scale="response",
    method="simulation",
    n_sim=2000,
    seed=42,
)

est.dydx("age")

GraphResult(estimate=array(0.00338638, dtype=float32), std_error=array(0.00413248, dtype=float32), conf_int_lower=array(-0.00482076, dtype=float32), conf_int_upper=array(0.01142533, dtype=float32), labels=['age'], method='simulation', level=0.95, ci='wald', scale='response', at='overall', plan=Plan(package_version='0.4.0', plan_hash='2fe6e60ec47480820d978890d1de65f723fad46c161b5bb241cce4f75bf62a43', node_kinds=('input',), node_hashes=('f13b864b9925fd888f42a7fdc04e2a33c397657905bb63c60b2e220f3f890cf6',), at='overall', scale='response', method_declared='simulation', method_resolved='simulation', method_resolution_reason='user-specified', vcov=None, ci='wald', level=0.95, B=1000, n_sim=2000, seed=42, gradient_backend='autodiff', fd_step=1e-06, data_fingerprint='8c8736cdd27606b38b7d39a20c868a8be138c0a4d60d8366fda79b8e19ca259e', weights_fingerprint=None, unhashable_callable=False, population_note=None, constants_overrides=()), population_note=None, n_obs=800, estimand_metadata={'kind': 'slo

In [3]:
from pymargins import steps, PysmatchClient
from pysmatch.Matcher import Matcher

test = df[df["treat"] == 1].copy()
control = df[df["treat"] == 0].copy()
matcher = Matcher(test, control, yvar="treat", exclude=["y"])
matcher.fit_scores(balance=True, model_type="linear")
matcher.predict_scores()
matcher.match(method="min", nmatches=1, threshold=0.001)

matched = matcher.matched_data
model = smf.logit("y ~ treat + x1 + age", data=matched).fit(disp=0)

est = GComputation(
    steps.match(steps.input(matched), PysmatchClient(matcher, treatment_col="treat")),
    outcome=model,
    at="overall",
    scale="response",
    method="bootstrap",
    B=999,
    seed=123,
)
est.contrasts(
    scenarios=[{"atexog": {"treat": 0}}, {"atexog": {"treat": 1}}],
    contrasts=[-1, 1],
)

2026-06-22 23:38:32 - INFO - Treatment column: treat


2026-06-22 23:38:32 - INFO - Covariates (xvars): ['age', 'x1']


2026-06-22 23:38:32 - INFO - Record ID source: generated index


2026-06-22 23:38:32 - INFO - N majority group (treatment=0): 605


2026-06-22 23:38:32 - INFO - N minority group (treatment=1): 395


...

2026-06-22 23:39:14 - INFO - Average Accuracy over 3 models: 45.93%


2026-06-22 23:39:14 - INFO - Propensity scores predicted and added to 'scores' column in self.data, self.test_df, self.control_df.


2026-06-22 23:39:14 - INFO - Performing matching using pysmatch.matching.perform_match: method='min', replacement=False, threshold=0.001, nmatches=1


2026-06-22 23:39:14 - INFO - Matching with pysmatch.matching.perform_match complete. Matched data has 634 rows.


GraphResult(estimate=array(0.06582421, dtype=float32), std_error=array(0.03373639, dtype=float32), conf_int_lower=array(-0.00366139, dtype=float32), conf_int_upper=array(0.12839472, dtype=float32), labels=['contrast'], method='bootstrap', level=0.95, ci='percentile', scale='response', at='overall', plan=Plan(package_version='0.4.0', plan_hash='b61d8785b1011c1ed365d5a6888e257f5f7baa85afc35838a623f6b8d93a748c', node_kinds=('input', 'match'), node_hashes=('e18f8257747c4cd26b1a34331b86a4d84b92f4ee7ae57c5d909ce833ee48ef06', 'd049a80c7de38a79db518088bf3087e1245c25966df66d742cb16a5672a7a867'), at='overall', scale='response', method_declared='bootstrap', method_resolved='bootstrap', method_resolution_reason='user-specified', vcov=None, ci='percentile', level=0.95, B=999, n_sim=4000, seed=123, gradient_backend='autodiff', fd_step=1e-06, data_fingerprint='7b0a16a760dc51dadc53ebfd937d5d53eb16953705996392c93ee15ad455d142', weights_fingerprint=None, unhashable_callable=False, population_note='match

In [4]:
print(est.plan.hash)
print(est.plan.describe())

b61d878@1
Plan b61d878@1
  method: bootstrap (declared: bootstrap)
  resolution reason: user-specified
  scale: response
  at: overall
  ci: percentile
  level: 0.95
  B: 999
  n_sim: 4000
  seed: 123
  data fingerprint: 7b0a16a760dc51da...
